# Your First Bandit in 2 Minutes

**Goal:** Run a complete multi-armed bandit simulation in under 2 minutes.

**What you'll learn:** How Thompson Sampling adaptively finds the best option by trying and learning.

**Scenario:** You have 3 commodity trading strategies. Which one should you use more often?

---

## Setup

Install the only two libraries we need:

In [ ]:
!pip install -q numpy matplotlib

## Quick Win: Create a 3-Arm Bandit

Each "arm" represents a trading strategy. The hidden reward probabilities are:
- Strategy A: 30% win rate
- Strategy B: 50% win rate
- Strategy C: 70% win rate (best, but we don't know this yet!)

The bandit will learn which is best by trying them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hidden truth: success rates for 3 trading strategies
true_rewards = [0.3, 0.5, 0.7]
n_arms = len(true_rewards)

print(f"Created {n_arms} trading strategies")
print(f"Hidden win rates: {true_rewards}")
print("\nGoal: Learn which strategy is best by trying them")

## Thompson Sampling in 8 Lines

Thompson Sampling uses a Beta distribution to track beliefs about each arm's reward.

**How it works:**
1. Start with no knowledge (Beta(1,1) for each arm)
2. Each round: sample from each arm's Beta distribution
3. Pick the arm with highest sample
4. Update that arm's Beta with the result

In [ ]:
# Thompson Sampling algorithm
np.random.seed(42)
n_rounds = 500

# Track beliefs: Beta(alpha, beta) for each arm
alpha = np.ones(n_arms)  # Successes + 1
beta = np.ones(n_arms)   # Failures + 1
pulls = np.zeros(n_arms)
rewards_history = []

for t in range(n_rounds):
    # Sample from each arm's posterior
    theta_samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
    
    # Pick arm with highest sample
    chosen_arm = np.argmax(theta_samples)
    pulls[chosen_arm] += 1
    
    # Get reward (simulate trading outcome)
    reward = 1 if np.random.random() < true_rewards[chosen_arm] else 0
    rewards_history.append(reward)
    
    # Update beliefs
    if reward == 1:
        alpha[chosen_arm] += 1
    else:
        beta[chosen_arm] += 1

print(f"Completed {n_rounds} rounds of trading")

## Results: Which Strategy Did We Learn to Prefer?

Count how many times each arm was pulled:

In [ ]:
print("\n=== Learning Results ===")
print(f"\nTotal rounds: {n_rounds}")
print(f"\nTimes each strategy was used:")
for i in range(n_arms):
    pct = 100 * pulls[i] / n_rounds
    print(f"  Strategy {chr(65+i)}: {int(pulls[i]):3d} times ({pct:.1f}%) - True win rate: {true_rewards[i]:.0%}")

best_arm = np.argmax(pulls)
print(f"\nMost selected: Strategy {chr(65+best_arm)} (the {'best' if best_arm == np.argmax(true_rewards) else 'WRONG'} choice!)")
print(f"Total wins: {sum(rewards_history)} out of {n_rounds} ({100*sum(rewards_history)/n_rounds:.1f}%)")

## Visualize: How Our Beliefs Evolved

Plot the Beta posterior for each arm. Taller, narrower distributions = more confident beliefs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.linspace(0, 1, 200)

for i in range(n_arms):
    # Plot posterior distribution
    from scipy.stats import beta as beta_dist
    y = beta_dist.pdf(x, alpha[i], beta[i])
    
    axes[i].plot(x, y, linewidth=2)
    axes[i].axvline(true_rewards[i], color='red', linestyle='--', label=f'True rate: {true_rewards[i]:.0%}')
    axes[i].set_title(f'Strategy {chr(65+i)}\n(pulled {int(pulls[i])} times)', fontsize=12)
    axes[i].set_xlabel('Win Rate', fontsize=10)
    axes[i].set_ylabel('Belief Density', fontsize=10)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Thompson Sampling Learned Beliefs After 500 Rounds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nNotice: The best strategy (C) has a narrow, confident peak near the true rate!")

## Modify This

Try these experiments:

1. **Change reward probabilities:** Make the strategies closer (e.g., [0.45, 0.48, 0.50]). Does it take longer to learn?
2. **Add more arms:** Try 5 strategies with `true_rewards = [0.2, 0.3, 0.5, 0.6, 0.7]`
3. **More rounds:** Change `n_rounds = 1000`. Do the beliefs get sharper?
4. **Start with knowledge:** Set `alpha = np.array([10, 5, 2])` to give Strategy A a head start

In [ ]:
# YOUR EXPERIMENTS HERE
# Copy the Thompson Sampling code from above and modify:

# Example: 5 strategies that are very close
# true_rewards = [0.45, 0.47, 0.48, 0.50, 0.49]
# n_arms = len(true_rewards)
# ... run the algorithm again ...

## What's Next?

You just ran Thompson Sampling, one of the most powerful bandit algorithms!

**Continue learning:**
- `01_ab_test_vs_bandit.ipynb` - See why bandits beat A/B tests
- `02_commodity_allocation_starter.ipynb` - Apply bandits to real commodity data
- `03_creator_bandit_playbook.ipynb` - Run the full playbook from the article
- `04_algorithm_comparison.ipynb` - Compare Thompson vs UCB vs ε-greedy

**Deep dive:**
- Module 1: Bandit fundamentals and regret analysis
- Module 2: Thompson Sampling theory and Beta-Bernoulli conjugacy
- Module 5: The two-wallet framework for commodity allocation

**Key insight:** Thompson Sampling automatically balances exploration and exploitation. No tuning needed!